# NB03c Centroidisation

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/CSBiology/BIO-BTE-06-L-7/gh-pages?filepath=NB03c_Centroidisation.ipynb)

[Download Notebook](https://github.com/CSBiology/BIO-BTE-06-L-7/releases/download/NB03a_NB03b_NB03c/NB03c_Centroidisation.ipynb)

1. Centroidisation
2. Peak fitting and picking functions
3. Application of the peak picking function
4. Questions

## Centroidisation

In reality, a peak is represented by a collection of signals from a peptide or fragment ion species that are measured by the 
specific detector. Due to imperfections of the measurement, there is a scatter around the accurate mass. This distribution 
along the m/z axis of signals from ion species is termed profile peak. The conversion of a peak profile into the corresponding m/z and 
intensity values reduces the complexity, its representation is termed centroiding. To extract the masses for identification in a simple 
and fast way, peak fitting approaches are used. Further, peak fitting algorithms are also needed to extract ion abundancies and therefore 
explained under quantification in the following section.


In [1]:
#r "nuget: BioFSharp, 2.0.0-beta5"
#r "nuget: BioFSharp.IO, 2.0.0-beta5"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: BioFSharp.Mz, 0.1.5-beta"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: MzIO, 0.1.7"
#r "nuget: MzIO.Processing, 0.1.2"
#r "nuget: MzLite, 1.1.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: MzIO.SQL, 0.1.9"

open Plotly.NET
open BioFSharp.Mz
open System.IO
open MzIO
open MzIO.Processing
open MzLite
open MzIO.MzSQL
open System.Data


Installed Packages BioFSharp, 2.0.0-beta5 BioFSharp.IO, 2.0.0-beta5 BioFSharp.Mz, 0.1.5-beta MzIO, 0.1.7 MzIO.Processing, 0.1.2 MzIO.SQL, 0.1.9 MzLite, 1.1.0 Plotly.NET, 4.2.0 Plotly.NET.Interactive, 4.2.0

Loading extensions from `/home/paulinehans/.nuget/packages/plotly.net.interactive/4.2.0/lib/netstandard2.1/Plotly.NET.Interactive.dll`

## Peak fitting and picking functions

We declare a function which centroids the given m/z and intensity data. In the scope of the function the m/z and intensity data 
are padded for the wavelet (You will read more about wavelet functions later in *NB05a\_Quantification.ipynb* ) 
and the centroided. For the centroidisation, we use a Ricker 2D wavelet.


In [2]:
// Code-Block 1

let ms1PeakPicking (mzData: float []) (intensityData: float []) = 
    if mzData.Length < 3 then 
        [||],[||]
    else
        let paddYValue = Array.min intensityData
        // we need to define some padding and wavelet parameters
        let paddingParams = 
            SignalDetection.Padding.createPaddingParameters paddYValue (Some 7) 0.05 150 95.
        let waveletParameters = 
            SignalDetection.Wavelet.createWaveletParameters 10 paddYValue 0.1 90. 1. false false
        
        let paddedMz,paddedIntensity = 
            SignalDetection.Padding.paddDataBy paddingParams mzData intensityData
        
        BioFSharp.Mz.SignalDetection.Wavelet.toCentroidWithRicker2D waveletParameters paddedMz paddedIntensity 


We load a sample MS1 from a mgf file.


In [3]:
// Code-Block 2
let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory; "downloads/WC_Gr3_UV64.mzlite"|]

//set up connection between file and SQL database
let runID = "sample=0"
let mzReader = new MzIO.MzSQL.MzSQL(path)
let peaksReader = new MzIO.MzSQL.MzSQL(path)
let cn = mzReader.Open()
let cn2 = peaksReader.Open()
let transaction = mzReader.BeginTransaction()


//function that returns all nessecary information we need, in this case m/z-ratio and intensity
let getMs2 = mzReader.ReadMassSpectra runID
let allSpectra = 
    getMs2
    |> Seq.choose (fun ms ->
        match MzIO.Processing.MassSpectrum.getMsLevel ms with
        | 1 -> 
            Some(
                MzIO.Processing.MassSpectrum.getScanTime ms,
                PeakArray.mzIntensityArrayOf (peaksReader.ReadSpectrumPeaks ms.ID)
            )
        | _ -> None
    )
    |> Seq.toArray

let allInfo = 
    allSpectra 
    |> Array.map (fun (rt,(mz,int)) -> rt, mz, int)
allInfo.[2222]


Item1,10.946383333333
Item2,"[ 350.01327903122217, 350.0310672739745, 350.0731838732097, 350.108960521994, 350.1572707903308, 350.16802031220425, 350.18517010102295, 350.203375433631, 350.22697295600227, 350.2519988527998, 350.28091166228063, 350.31759209239215, 350.3478507763394, 350.6674065753155, 350.6950215168385, 350.72395260549507, 350.80753804031025, 350.8283049336639, 350.8626299357615, 350.88522263960573 ... (7188 more) ]"
Item3,"[ 4.729430973031867, 38.75639197070484, 19.839221832623366, 38.103876952010296, 7.0956509142718005, 33.77074794352666, 13.666308517622383, 9.06727749010372, 24.18020990352545, 11.827726887503331, 2.3656430118364824, 38.377871271254094, 14.721020044805755, 7.100843280386471, 17.358300347166278, 9.46855436281453, 7.102254538875741, 15.125682063933255, 50.37715783719534, 38.80377169490998 ... (7188 more) ]"


## Application of the peak picking function

We centroid the MS2 data using the function declared beforehand:


In [4]:
// Code-Block 3

let mzData = allInfo.[100] |> fun (rt,mz,int) -> mz
let intensity = allInfo.[100] |> fun (rt,mz,int) -> int

let centroidedMs1 = 
    ms1PeakPicking mzData intensity
centroidedMs1

Item1,"[ 350.02933201166803, 350.8325193660877, 351.2385544964327, 351.86450457098607, 352.23769283402373, 352.85116335808567, 353.3226025136863, 353.91256093413836, 354.0784310447636, 354.84245221827337, 355.06870309044916, 355.31846050706025, 355.87418570798707, 356.0687056110666, 356.278577913984, 356.8511582981012, 357.0669315472895, 357.2864751773779, 357.85066699203884, 358.0699052177006 ... (1273 more) ]"
Item2,"[ 35.73482173336231, 83.78375756482797, 111.4679847387606, 87.46287809742341, 76.30653630822053, 55.92806376511987, 37.61821037584809, 33.81857918419655, 51.664263539584, 62.03768826107137, 3042.5647912504446, 39.17995013693201, 65.04247825410232, 1036.9909489618522, 160.77632422376666, 90.60076630929075, 1073.476538310291, 59.06542250117184, 35.86597029317602, 349.201469147593 ... (1273 more) ]"


In [5]:
let zip = Array.zip mzData intensity
let filteredMs1 = 
    zip 
    |> Array.filter (fun (mass, intensity) ->
        intensity > 4000.
    )
    |> Array.unzip

let filteredChart = 
    [
        Chart.Point(fst filteredMs1, snd filteredMs1 )
        |> Chart.withTraceInfo "Uncentroided MS1"
        Chart.Point(fst centroidedMs1, snd centroidedMs1)
        |> Chart.withTraceInfo "Centroided MS1"

    ]
    |> Chart.combine
    |> Chart.withYAxisStyle "Intensity"
    |> Chart.withXAxisStyle (TitleText = "m/z", MinMax = (400., 800.))
    |> Chart.withSize (900., 900.)

filteredChart


<!-- Plotly chart will be drawn inside this DIV -->

## Questions:

1. The aim of centroidization is finding the m/z for each profile peak. How can this improve the performance and quality of the following steps?
2. In the result plot, a single ms1 spectrum is shown. Naively describe the differences between the uncentroided and the centroided spectrums.
3. Taking into consideration your answer for question 1, do your findings of question 2 meet your expectations? If yes, why? If no, why?

